# EOS projections


In [ ]:
## Read EOS projection
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.chdir(Path("/Users/xingyihuang/Jupyter Code/Phenology/Appeal/figures/main"))
OUT_DIR = Path("../../results/figure4")
OUT_DIR.mkdir(parents=True, exist_ok=True)
EVAL6_DIR = Path("../../results/model/projection/eval6")

MODEL_ORDER = ["CDD", "CDDP", "SIAM", "SIAMP"]
MODEL_COLORS = {
    "CDD": "#a3d4e1",
    "CDDP": "#5cadd8",
    "SIAM": "#ea9e87",
    "SIAMP": "#cf847e",
}

def read_eos_projection(ssp, start_year=2026, end_year=2100):
        fp = EVAL6_DIR / f"eval6_{ssp}_predictions.pkl"
    preds = pd.read_pickle(fp)
    out = []
    for p in preds:
        years = np.asarray(p["years"], int)
        eos = np.asarray(p["predicted_eos"], float)
        keep = (years >= start_year) & (years <= end_year)
        q = dict(p)
        q["years"] = years[keep]
        q["predicted_eos"] = eos[keep]
        out.append(q)
    print(f"Loaded {fp} | n={len(out)} | grids={len({(r['latitude'], r['longitude']) for r in out})}")
    print(f"models={sorted({r['model'] for r in out})} | years={out[0]['years'][0]}–{out[0]['years'][-1]}")
    return out


In [ ]:
## Filter out all-365 EOS series
def filter_data_by_coordinates(data):
    coordinates = {(d["latitude"], d["longitude"]) for d in data}
    bad = set()
    for coord in coordinates:
        for m in (d for d in data if (d["latitude"], d["longitude"]) == coord):
            if np.all(np.asarray(m["predicted_eos"]) == 365):
                bad.add(coord)
                break
    clean = [d for d in data if (d["latitude"], d["longitude"]) not in bad]
    print(f"Kept {len(clean)} / {len(data)} records ({len(coordinates) - len(bad)} grids)")
    return clean


In [ ]:
def plot_eos_trend(all_predictions, ymax=None):
    from matplotlib.ticker import MultipleLocator, FixedLocator

    years = np.asarray(all_predictions[0]["years"], int)
    start_year, end_year = int(years.min()), int(years.max())

    df_preds = pd.DataFrame({
        "latitude": [p["latitude"] for p in all_predictions],
        "longitude": [p["longitude"] for p in all_predictions],
        "model": [p["model"] for p in all_predictions],
        "eos": [p["predicted_eos"] for p in all_predictions],
    })
    models = [m for m in MODEL_ORDER if m in set(df_preds["model"])]

    fig, ax_main = plt.subplots(figsize=(6, 5))
    means_all = []
    for model in models:
        model_rows = df_preds[df_preds["model"] == model]
        eos_matrix = np.stack(model_rows["eos"].values)
        eos_df = pd.DataFrame(eos_matrix, columns=years)
        mean = eos_df.mean(axis=0)
        std = eos_df.std(axis=0)
        means_all.append(mean.to_numpy(dtype=float))
        color = MODEL_COLORS.get(model, "black")
        ax_main.plot(years, mean, label=model, color=color, linewidth=2)
        ax_main.fill_between(
            years, mean - std / 4, mean + std / 4, color="lightgray", alpha=0.2
        )

    ax_main.set_xlabel("Year", fontsize=16)
    ax_main.set_ylabel("EOS (DOY)", fontsize=16)
    ax_main.set_xlim(start_year, end_year)
    eos_lo = float(np.nanmin(means_all))
    eos_hi = float(np.nanmax(means_all))
    y0 = eos_lo - 5.0
    y1 = float(ymax) if ymax is not None else eos_hi + 5.0
    ax_main.set_ylim(y0, y1)
    ax_main.yaxis.set_major_locator(MultipleLocator(10))
    ax_main.xaxis.set_major_locator(FixedLocator([2040, 2060, 2080, 2100]))
    ax_main.tick_params(axis="both", labelsize=14)
    plt.subplots_adjust(right=0.95, bottom=0.15)

    fig_delay, ax_inset = plt.subplots(figsize=(6, 5))
    diff_means, diff_errors = [], []
    for model in models:
        model_rows = df_preds[df_preds["model"] == model]
        eos_df = pd.DataFrame(np.stack(model_rows["eos"].values), columns=years)
        first = eos_df.loc[:, start_year:start_year + 4].mean(axis=1)
        last = eos_df.loc[:, end_year - 4:end_year].mean(axis=1)
        d = last - first
        diff_means.append(float(d.mean()))
        diff_errors.append(0.25 * float(d.std(ddof=1)))

    print("EOS delays by 2100:", {m: round(v, 2) for m, v in zip(models, diff_means)})
    x = np.arange(len(models))
    ax_inset.bar(
        x, diff_means, yerr=diff_errors, capsize=4,
        color=[MODEL_COLORS[m] for m in models],
    )
    ax_inset.set_xticks(x)
    ax_inset.set_xticklabels(models, rotation=45, ha="right", fontsize=28)
    ax_inset.set_ylabel("EOS delay (days)", fontsize=30)
    ax_inset.axhline(0, color="black", linewidth=0.5, linestyle="--")
    dmin = float(np.nanmin(diff_means))
    dmax = float(np.nanmax(diff_means))
    ax_inset.set_ylim(dmin - 2.0, dmax + 2.0)
    ax_inset.yaxis.set_major_locator(MultipleLocator(5))
    ax_inset.tick_params(axis="both", labelsize=28)
    fig_delay.tight_layout()

    return fig, fig_delay



## SSP245


In [ ]:
eos_projection_ssp245 = read_eos_projection("ssp245")
clean_eos_projection = filter_data_by_coordinates(eos_projection_ssp245)
fig, fig_delay = plot_eos_trend(clean_eos_projection, ymax=316)
fp = OUT_DIR / "ssp245_all_models.png"
fp_delay = OUT_DIR / "ssp245_eos_delay.png"
fig.savefig(fp, dpi=500, bbox_inches="tight")
fig_delay.savefig(fp_delay, dpi=500, bbox_inches="tight")
plt.show()
print("Saved", fp)
print("Saved", fp_delay)



## SSP585


In [ ]:
eos_projection_ssp585 = read_eos_projection("ssp585")
clean_eos_projection = filter_data_by_coordinates(eos_projection_ssp585)
fig, fig_delay = plot_eos_trend(clean_eos_projection, ymax=325)
fp = OUT_DIR / "ssp585_all_models.png"
fp_delay = OUT_DIR / "ssp585_eos_delay.png"
fig.savefig(fp, dpi=500, bbox_inches="tight")
fig_delay.savefig(fp_delay, dpi=500, bbox_inches="tight")
plt.show()
print("Saved", fp)
print("Saved", fp_delay)

